# 01 — Data Validation and Cleaning

*Executed analytical companion to the validated production pipeline.*

## Objective

Document the validated ingestion and cleaning contract using existing profiles and processed Parquet metadata. This notebook does **not** open or rewrite the raw Excel workbook.

## Business Questions

- What fields, worksheets, and date coverage are present in the source?
- Where are missing values and exact duplicates concentrated?
- How are completed sales, returns, anonymous sales, and excluded records separated?
- Do row counts and monetary values reconcile after cleaning?

## Inputs

- `reports/raw_data_profile.json`
- `reports/cleaning_summary.json`
- Processed Parquet metadata under `data/processed/`
- Reusable contracts in `retail_analytics.cleaning`

## Methodology

Read compact validation artifacts and Parquet schemas only. The production functions remain the source of truth; no raw-workbook ingestion or cleaning is rerun here.

## Code

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "src" / "retail_analytics").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 14)
pd.set_option("display.width", 140)

def load_json(relative_path):
    return json.loads((PROJECT_ROOT / relative_path).read_text(encoding="utf-8"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")

import pyarrow.parquet as pq
from retail_analytics.cleaning import (
    calculate_outlier_thresholds,
    partition_masks,
    prepare_transactions,
)

raw_profile = load_json("reports/raw_data_profile.json")
cleaning = load_json("reports/cleaning_summary.json")
print("Reusable cleaning module loaded:", prepare_transactions.__module__)

Project root: C:\Projects\Retail-Customer-Intelligence-Analytics
Python: 3.12.7
Reusable cleaning module loaded: retail_analytics.cleaning


In [2]:
raw_schema = pd.DataFrame(
    [
        ("Invoice", "string", "Transaction identifier; C prefix indicates cancellation"),
        ("StockCode", "string", "Product identifier"),
        ("Description", "nullable string", "Product description"),
        ("Quantity", "integer", "Units; negative values indicate returns"),
        ("InvoiceDate", "datetime", "Transaction timestamp"),
        ("Price", "numeric", "Unit price in GBP"),
        ("Customer ID", "nullable string", "Customer identifier"),
        ("Country", "string", "Customer country"),
    ],
    columns=["Raw field", "Expected type", "Meaning"],
)
display(raw_schema)
display(pd.DataFrame({
    "Source worksheet": raw_profile["coverage"]["source_sheets"],
    "Role": ["First source year", "Second source year"],
}))
print(
    "Coverage:", raw_profile["coverage"]["minimum_invoice_date"], "to",
    raw_profile["coverage"]["maximum_invoice_date"]
)

,Raw field,Expected type,Meaning
0,Invoice,string,Transaction identifier; C prefix indicates can...
1,StockCode,string,Product identifier
2,Description,nullable string,Product description
3,Quantity,integer,Units; negative values indicate returns
4,InvoiceDate,datetime,Transaction timestamp
5,Price,numeric,Unit price in GBP
6,Customer ID,nullable string,Customer identifier
7,Country,string,Customer country


,Source worksheet,Role
0,Year 2009-2010,First source year
1,Year 2010-2011,Second source year


Coverage: 2009-12-01T07:45:00 to 2011-12-09T12:50:00


In [3]:
missing = (
    pd.DataFrame(raw_profile["missing_values"]).T
    .rename_axis("field").reset_index()
    .rename(columns={"count": "missing_rows", "percent": "missing_pct"})
)
missing["missing_rows"] = missing["missing_rows"].map(lambda value: f"{value:,.0f}")
missing["missing_pct"] = missing["missing_pct"].map(lambda value: f"{value:.2f}%")
display(missing)

duplicates = pd.DataFrame([
    ("Rows read", raw_profile["row_counts"]["total"]),
    ("Exact duplicates within sheets", raw_profile["row_counts"]["exact_duplicates_within_sheet"]),
    ("Exact duplicates across workbook", raw_profile["row_counts"]["exact_duplicates_across_workbook"]),
    ("Unique cleaned rows", cleaning["unique_rows"]),
], columns=["Audit measure", "Rows"])
duplicates["Rows"] = duplicates["Rows"].map(lambda value: f"{value:,.0f}")
display(duplicates)

,field,missing_rows,missing_pct
0,description,"4,382",0.41%
1,customer_id,"243,007",22.77%


,Audit measure,Rows
0,Rows read,"1,067,371"
1,Exact duplicates within sheets,"12,133"
2,Exact duplicates across workbook,"34,335"
3,Unique cleaned rows,"1,033,036"


In [4]:
parquet_path = PROJECT_ROOT / "data" / "processed" / "completed_sales.parquet"
parquet = pq.ParquetFile(parquet_path)
processed_schema = pd.DataFrame(
    [(field.name, str(field.type)) for field in parquet.schema_arrow],
    columns=["Processed field", "Parquet type"],
)
print(f"Completed-sales rows from Parquet metadata: {parquet.metadata.num_rows:,.0f}")
display(processed_schema.head(14))

Completed-sales rows from Parquet metadata: 1,007,913


,Processed field,Parquet type
0,transaction_line_id,int64
1,invoice_no,string
2,stock_code,string
3,description,string
4,quantity,int64
...,...,...
9,source_sheet,string
10,line_revenue,double
11,is_cancelled,bool
12,is_return,bool


## Results

The following tables reproduce the validated cleaning decisions and reconciliation—not a second implementation of them.

In [5]:
decisions = pd.DataFrame([
    ("Exact duplicates", "Remove exact repeated source rows once"),
    ("Completed sales", "Positive quantity and price; invoice is not cancelled"),
    ("Returns/cancellations", "Cancelled invoice or negative quantity"),
    ("Anonymous sales", "Retain for aggregate sales; exclude from customer analytics"),
    ("Non-sales records", "Isolate with an explicit exclusion reason"),
    ("Extreme values", "Flag with IQR thresholds; do not automatically delete"),
], columns=["Decision area", "Validated treatment"])
display(decisions)

partitions = pd.DataFrame(
    [(name.replace("_", " ").title(), value) for name, value in cleaning["partitions"].items()],
    columns=["Partition", "Rows"],
)
partitions["Rows"] = partitions["Rows"].map(lambda value: f"{value:,.0f}")
display(partitions)

reconciliation = pd.DataFrame([
    ("Unique rows", cleaning["unique_rows"]),
    ("Partition total", cleaning["reconciliation"]["partition_total"]),
    ("Completed-sales value", f"GBP {cleaning['completed_sales_value_gbp']:,.2f}"),
    ("Signed returns value", f"GBP {cleaning['returns_value_gbp']:,.2f}"),
], columns=["Reconciliation measure", "Validated result"])
display(reconciliation)
print("Partition reconciliation passed:", cleaning["reconciliation"]["matches_unique_rows"])

,Decision area,Validated treatment
0,Exact duplicates,Remove exact repeated source rows once
1,Completed sales,Positive quantity and price; invoice is not ca...
2,Returns/cancellations,Cancelled invoice or negative quantity
3,Anonymous sales,Retain for aggregate sales; exclude from custo...
4,Non-sales records,Isolate with an explicit exclusion reason
5,Extreme values,Flag with IQR thresholds; do not automatically...


,Partition,Rows
0,Completed Sales,"1,007,913"
1,Returns Or Cancellations,"22,497"
2,Excluded Non Sales,"2,626"
3,Customer Sales,"779,425"
4,Anonymous Sales,"228,488"


,Reconciliation measure,Validated result
0,Unique rows,1033036
1,Partition total,1033036
2,Completed-sales value,"GBP 20,476,260.45"
3,Signed returns value,"GBP -1,462,050.61"


Partition reconciliation passed: True


## Business Insights

- The two worksheets cover 1 December 2009 through 9 December 2011; the last month is partial.
- Missing customer IDs are expected and materially important: anonymous completed sales remain in aggregate revenue, but cannot support customer-level RFM, cohort, statistical, or clustering analysis because no stable customer key exists.
- Exact duplicate removal prevents double counting while preserving legitimate repeated product lines.
- Outliers are flagged rather than deleted because they may represent genuine wholesale or high-volume activity; deletion would silently change confirmed revenue.

## Assumptions

- Invoice prefixes and quantity signs retain their documented transactional meaning.
- GBP is the reporting currency and no foreign-exchange conversion is required.
- Existing JSON reports and Parquet files are outputs of the validated production pipeline.

## Limitations

- The source has no unit cost, margin, marketing exposure, or demographic fields.
- Missing descriptions limit product interpretation for a small subset of rows.
- Validation identifies unusual observations but cannot independently confirm whether each is wholesale, error, or another operational case.

## Next Steps

- Monitor future source extracts against the same schema and reconciliation gates.
- Investigate flagged high-value lines with operational context before any exclusion decision.
- Preserve worksheet and transaction-line lineage in downstream refreshes.